In [ ]:
# Environment Setup and Library Imports
import sys
import os
from pathlib import Path
import importlib

# Configure project root for module imports
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Data manipulation and analysis
import pandas as pd
import numpy as np

# Visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Reload modules for latest functions
import src.analysis
importlib.reload(src.analysis)

from src.loader import load_data, clean_data
from src.analysis import (calculate_driver_stats, calculate_team_stats,
                          calculate_combined_standings, calculate_combined_constructor_standings)
from src.features import prepare_features
from src.model import train_model, load_trained_model
from src.evaluate import evaluate_model

plt.style.use('seaborn-v0_8-darkgrid')
print('Environment configured successfully.')

In [ ]:
# =============================================================================
# ENVIRONMENT SETUP
# =============================================================================
import sys
import os
from pathlib import Path
import pickle
import warnings
warnings.filterwarnings('ignore')

# Configure project root
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Data and ML libraries
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ML libraries
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Project imports
try:
    from src.loader import load_data, clean_data
    from src.analysis import (
        calculate_driver_stats, 
        calculate_team_stats,
        calculate_combined_standings, 
        calculate_combined_constructor_standings
    )
    from src.config import TEAM_COLORS, get_team_color, MODEL_CONFIG
except ImportError as e:
    print(f'Import warning: {e}')
    TEAM_COLORS = {}
    def get_team_color(team): return '#808080'

# Paths
MODEL_DIR = project_root / 'models'
MODEL_DIR.mkdir(exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
print('Environment configured successfully.')
print(f'Model directory: {MODEL_DIR}')

## 1. Load Season Data

In [ ]:
# =============================================================================
# LOAD RACE AND SPRINT DATA
# =============================================================================
race_path = project_root / 'data' / 'Formula1_2025Season_RaceResults.csv'
sprint_path = project_root / 'data' / 'Formula1_2025Season_SprintResults.csv'

try:
    df_race = load_data(str(race_path))
    df_sprint = load_data(str(sprint_path))
    
    if df_race is None:
        raise FileNotFoundError(f'Race data not found at {race_path}')
    
    df_race = clean_data(df_race)
    
    # Fix sprint data types
    if df_sprint is not None:
        df_sprint['Points'] = pd.to_numeric(df_sprint['Points'], errors='coerce').fillna(0)
        df_sprint['Position'] = pd.to_numeric(df_sprint['Position'], errors='coerce')
    
    print('=' * 60)
    print('           DATA LOADED SUCCESSFULLY')
    print('=' * 60)
    print(f'Race entries: {len(df_race)} from {df_race["Track"].nunique()} Grand Prix')
    if df_sprint is not None:
        print(f'Sprint entries: {len(df_sprint)} from {df_sprint["Track"].nunique()} events')
    print(f'Drivers: {df_race["Driver"].nunique()}')
    print(f'Teams: {df_race["Team"].nunique()}')
    
except Exception as e:
    print(f'ERROR loading data: {e}')
    df_race = None
    df_sprint = None

## 2. Current Championship Standings

In [ ]:
# =============================================================================
# CHAMPIONSHIP STANDINGS (RACE + SPRINT)
# =============================================================================
if df_race is not None:
    try:
        combined_driver_standings = calculate_combined_standings(df_race, df_sprint)
        combined_team_standings = calculate_combined_constructor_standings(df_race, df_sprint)
        
        print('=' * 70)
        print("         2025 FIA FORMULA 1 DRIVERS' WORLD CHAMPIONSHIP")
        print('         (Race + Sprint Combined Points)')
        print('=' * 70)
        
        display(combined_driver_standings.head(10))
        
    except Exception as e:
        print(f'Error calculating standings: {e}')
        combined_driver_standings = None
        combined_team_standings = None

In [ ]:
# =============================================================================
# CONSTRUCTOR STANDINGS
# =============================================================================
if combined_team_standings is not None:
    print('=' * 70)
    print("         2025 FIA FORMULA 1 CONSTRUCTORS' CHAMPIONSHIP")
    print('=' * 70)
    display(combined_team_standings)

## 3. Model Training

In [ ]:
# =============================================================================
# PREPARE FEATURES AND TRAIN MODEL
# =============================================================================
if df_race is not None:
    # Filter to finished races only
    df_train = df_race[df_race['Finished'] == True].copy()
    print(f'Training samples: {len(df_train)} (finished races only)')
    
    # Features and target
    features = ['Starting Grid', 'Driver', 'Team', 'Track']
    target = 'Position'
    
    X = df_train[features].copy()
    y = df_train[target].copy()
    
    # Encode categorical variables
    encoders = {}
    for col in ['Driver', 'Team', 'Track']:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col])
        encoders[col] = le
        
        # Save encoder to models/ directory
        encoder_path = MODEL_DIR / f'{col}_encoder.pkl'
        with open(encoder_path, 'wb') as f:
            pickle.dump(le, f)
        print(f'Saved: {encoder_path.name} ({len(le.classes_)} classes)')
    
    # Train/test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    print(f'\nTrain size: {len(X_train)}, Test size: {len(X_test)}')

In [ ]:
# =============================================================================
# TRAIN RANDOM FOREST MODEL
# =============================================================================
if df_race is not None:
    print('Training Random Forest model...')
    
    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=10,
        min_samples_split=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)
    
    # Save model to models/ directory
    model_path = MODEL_DIR / 'f1_model.pkl'
    with open(model_path, 'wb') as f:
        pickle.dump(model, f)
    print(f'Model saved to: {model_path}')
    
    # Evaluate
    y_pred = model.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_absolute_error')
    cv_mae = -cv_scores.mean()
    cv_std = cv_scores.std()
    
    print('\n' + '=' * 60)
    print('              MODEL PERFORMANCE METRICS')
    print('=' * 60)
    print(f'''
  Mean Absolute Error (MAE):     {mae:.2f} positions
  Root Mean Squared Error:       {rmse:.2f} positions
  R-squared Score:               {r2:.3f}
  
  Cross-Validation MAE:          {cv_mae:.2f} +/- {cv_std:.2f} positions
  Model Accuracy:                {(1 - cv_mae/10) * 100:.1f}%
  Prediction Confidence:         +/-{cv_mae + cv_std:.1f} positions (95% CI)
''')
    print('=' * 60)

In [ ]:
# =============================================================================
# FEATURE IMPORTANCE
# =============================================================================
if df_race is not None:
    importance = pd.DataFrame({
        'Feature': features,
        'Importance': model.feature_importances_
    }).sort_values('Importance', ascending=True)
    
    fig = go.Figure(go.Bar(
        x=importance['Importance'],
        y=importance['Feature'],
        orientation='h',
        marker_color=['#DC0000', '#FF8700', '#00D2BE', '#1E41FF'],
        text=[f'{v:.1%}' for v in importance['Importance']],
        textposition='outside'
    ))
    
    fig.update_layout(
        title='Feature Importance for Position Prediction',
        xaxis_title='Importance',
        template='plotly_dark',
        height=400
    )
    fig.show()

In [ ]:
# =============================================================================
# PREDICTION VS ACTUAL
# =============================================================================
if df_race is not None:
    results = pd.DataFrame({
        'Actual': y_test.values,
        'Predicted': y_pred
    })
    
    fig = go.Figure()
    
    fig.add_trace(go.Scatter(
        x=results['Actual'],
        y=results['Predicted'],
        mode='markers',
        marker=dict(color='#FF8700', size=8, opacity=0.6),
        name='Predictions'
    ))
    
    # Ideal line
    fig.add_trace(go.Scatter(
        x=[1, 20], y=[1, 20],
        mode='lines',
        line=dict(color='red', dash='dash'),
        name='Perfect Prediction'
    ))
    
    fig.update_layout(
        title='Model Validation: Predicted vs Actual Positions',
        xaxis_title='Actual Position',
        yaxis_title='Predicted Position',
        template='plotly_dark',
        height=500
    )
    fig.show()

## 4. Abu Dhabi GP Prediction

In [ ]:
# =============================================================================
# GENERATE PREDICTED STARTING GRID
# =============================================================================
if df_race is not None:
    # Get unique drivers and their teams
    drivers_teams = df_race.groupby('Driver')['Team'].first().reset_index()
    
    # Calculate average grid position as proxy for qualifying
    avg_grid = df_race.groupby('Driver')['Starting Grid'].mean().reset_index()
    avg_grid.columns = ['Driver', 'Avg_Grid']
    
    # Merge and sort by predicted grid
    abu_dhabi_grid = drivers_teams.merge(avg_grid, on='Driver')
    abu_dhabi_grid = abu_dhabi_grid.sort_values('Avg_Grid').reset_index(drop=True)
    abu_dhabi_grid['Predicted_Grid'] = range(1, len(abu_dhabi_grid) + 1)
    
    print('=' * 70)
    print('         ABU DHABI GP 2025 - PREDICTED STARTING GRID')
    print('         Yas Marina Circuit - December 7, 2025')
    print('=' * 70)
    display(abu_dhabi_grid[['Predicted_Grid', 'Driver', 'Team', 'Avg_Grid']].head(20))

In [ ]:
# =============================================================================
# GENERATE RACE PREDICTIONS
# =============================================================================
if df_race is not None:
    # Prepare prediction data
    abu_dhabi_pred = abu_dhabi_grid.copy()
    abu_dhabi_pred['Starting Grid'] = abu_dhabi_pred['Predicted_Grid']
    
    # Determine track encoding
    if 'Abu Dhabi' in encoders['Track'].classes_:
        abu_dhabi_pred['Track'] = 'Abu Dhabi'
    else:
        # Use similar track as proxy
        print('Note: Abu Dhabi not in training data, using proxy encoding')
        abu_dhabi_pred['Track'] = encoders['Track'].classes_[0]
    
    # Prepare features
    X_pred = abu_dhabi_pred[['Starting Grid', 'Driver', 'Team', 'Track']].copy()
    
    # Encode with error handling
    for col in ['Driver', 'Team', 'Track']:
        le = encoders[col]
        X_pred[col] = X_pred[col].apply(
            lambda x: le.transform([x])[0] if x in le.classes_ else 0
        )
    
    # Generate ensemble predictions for confidence intervals
    all_predictions = np.array([tree.predict(X_pred) for tree in model.estimators_])
    predicted_positions = all_predictions.mean(axis=0)
    prediction_std = all_predictions.std(axis=0)
    
    # Add predictions to dataframe
    abu_dhabi_pred['Predicted_Position'] = predicted_positions
    abu_dhabi_pred['Prediction_Std'] = prediction_std
    abu_dhabi_pred['Confidence_Lower'] = np.maximum(1, predicted_positions - 1.96 * prediction_std)
    abu_dhabi_pred['Confidence_Upper'] = np.minimum(20, predicted_positions + 1.96 * prediction_std)
    
    # Sort by predicted position
    final_prediction = abu_dhabi_pred.sort_values('Predicted_Position').reset_index(drop=True)
    final_prediction['Final_Position'] = range(1, len(final_prediction) + 1)
    
    print('Predictions generated with 95% confidence intervals.')

In [ ]:
# =============================================================================
# PREDICTED RACE RESULTS
# =============================================================================
if df_race is not None:
    # Points system
    points_system = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}
    
    final_prediction['Predicted_Points'] = final_prediction['Final_Position'].map(
        lambda x: points_system.get(x, 0)
    )
    
    # Confidence percentage
    final_prediction['Confidence_%'] = (100 - final_prediction['Prediction_Std'] * 10).clip(50, 99).round(1)
    
    print('=' * 80)
    print('                 ABU DHABI GP 2025 - PREDICTED RACE RESULTS')
    print('                 Yas Marina Circuit - December 7, 2025')
    print('=' * 80)
    
    result_display = final_prediction[[
        'Final_Position', 'Driver', 'Team', 'Predicted_Grid', 
        'Predicted_Points', 'Confidence_%'
    ]].copy()
    result_display.columns = ['Pos', 'Driver', 'Team', 'Grid', 'Points', 'Confidence']
    display(result_display)

In [ ]:
# =============================================================================
# VISUALIZE TOP 10 PREDICTION
# =============================================================================
if df_race is not None:
    top_10 = final_prediction.head(10)
    
    # Get team colors
    colors = [get_team_color(t) for t in top_10['Team']]
    
    fig = go.Figure()
    
    fig.add_trace(go.Bar(
        x=top_10['Driver'],
        y=top_10['Predicted_Points'],
        marker_color=colors,
        text=[f"P{p}\n{c}%" for p, c in zip(top_10['Final_Position'], top_10['Confidence_%'])],
        textposition='outside',
        error_y=dict(
            type='data',
            array=top_10['Prediction_Std'] * 2,
            visible=True,
            color='white'
        )
    ))
    
    fig.update_layout(
        title='Abu Dhabi GP 2025 - Predicted Top 10',
        xaxis_title='Driver',
        yaxis_title='Predicted Points',
        template='plotly_dark',
        height=500,
        showlegend=False
    )
    fig.show()

## 5. Championship Impact Analysis

In [ ]:
# =============================================================================
# PREDICTED FINAL CHAMPIONSHIP STANDINGS
# =============================================================================
if df_race is not None and combined_driver_standings is not None:
    # Current standings
    current_standings = combined_driver_standings[['Total_Points', 'Race_Points', 'Sprint_Points']].copy()
    current_standings = current_standings.reset_index()
    
    # Merge with Abu Dhabi predictions
    abu_dhabi_points = final_prediction[['Driver', 'Predicted_Points', 'Confidence_%']]
    final_standings = current_standings.merge(abu_dhabi_points, on='Driver', how='left')
    final_standings['Predicted_Points'] = final_standings['Predicted_Points'].fillna(0)
    
    # Calculate final totals
    final_standings['Final_Total'] = final_standings['Total_Points'] + final_standings['Predicted_Points']
    final_standings = final_standings.sort_values('Final_Total', ascending=False).reset_index(drop=True)
    final_standings['Final_Position'] = range(1, len(final_standings) + 1)
    
    # Position change
    current_pos = {d: i+1 for i, d in enumerate(
        current_standings.sort_values('Total_Points', ascending=False)['Driver']
    )}
    final_standings['Position_Change'] = final_standings['Driver'].map(current_pos) - final_standings['Final_Position']
    
    print('=' * 85)
    print('         PREDICTED FINAL 2025 CHAMPIONSHIP STANDINGS')
    print('         (After Abu Dhabi GP)')
    print('=' * 85)
    
    display_df = final_standings[[
        'Final_Position', 'Driver', 'Total_Points', 'Predicted_Points', 'Final_Total', 'Position_Change'
    ]].head(15).copy()
    display_df.columns = ['Pos', 'Driver', 'Current Pts', 'Abu Dhabi', 'Final Total', 'Change']
    display_df['Change'] = display_df['Change'].apply(
        lambda x: f'+{int(x)}' if x > 0 else (f'{int(x)}' if x < 0 else '-')
    )
    display(display_df)

In [ ]:
# =============================================================================
# CHAMPIONSHIP BREAKDOWN VISUALIZATION
# =============================================================================
if df_race is not None:
    top_8 = final_standings.head(8)
    
    fig = go.Figure()
    
    # Race points
    fig.add_trace(go.Bar(
        name='Race Points',
        x=top_8['Driver'],
        y=top_8['Race_Points'],
        marker_color='#1E1E1E',
        text=top_8['Race_Points'].astype(int),
        textposition='inside'
    ))
    
    # Sprint points
    fig.add_trace(go.Bar(
        name='Sprint Points',
        x=top_8['Driver'],
        y=top_8['Sprint_Points'],
        marker_color='#FF8700',
        text=top_8['Sprint_Points'].astype(int),
        textposition='inside'
    ))
    
    # Abu Dhabi predicted
    fig.add_trace(go.Bar(
        name='Abu Dhabi (Predicted)',
        x=top_8['Driver'],
        y=top_8['Predicted_Points'],
        marker_color='#DC0000',
        text=top_8['Predicted_Points'].astype(int),
        textposition='inside'
    ))
    
    fig.update_layout(
        title='Predicted Final Championship - Points Breakdown',
        barmode='stack',
        template='plotly_dark',
        yaxis_title='Total Points',
        legend=dict(orientation='h', yanchor='bottom', y=1.02),
        height=500
    )
    fig.show()

## 6. Prediction Summary

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================
if df_race is not None:
    winner = final_prediction.iloc[0]
    p2 = final_prediction.iloc[1]
    p3 = final_prediction.iloc[2]
    champion = final_standings.iloc[0]
    
    print('=' * 70)
    print('             ABU DHABI GP 2025 - PREDICTION SUMMARY')
    print('=' * 70)
    print(f'''
    PREDICTED PODIUM:
    -----------------
    1st: {winner['Driver']:<25} ({winner['Team']})
         Confidence: {winner['Confidence_%']:.1f}%

    2nd: {p2['Driver']:<25} ({p2['Team']})
         Confidence: {p2['Confidence_%']:.1f}%

    3rd: {p3['Driver']:<25} ({p3['Team']})
         Confidence: {p3['Confidence_%']:.1f}%

    2025 WORLD CHAMPION PREDICTION:
    --------------------------------
    {champion['Driver']}
    Final Points: {champion['Final_Total']:.0f}
    (Current: {champion['Total_Points']:.0f} + Abu Dhabi: {champion['Predicted_Points']:.0f})

    MODEL ACCURACY METRICS:
    -----------------------
    Mean Absolute Error:      {mae:.2f} positions
    R-squared Score:          {r2:.3f}
    Cross-Validation MAE:     {cv_mae:.2f} +/- {cv_std:.2f}
    Overall Model Accuracy:   {(1 - cv_mae/10) * 100:.1f}%
''')
    print('=' * 70)
    print('Models saved to:', MODEL_DIR)
    print('=' * 70)

---

## Methodology

### Model Architecture
- **Algorithm:** Random Forest Regressor (100 trees)
- **Features:** Starting Grid, Driver, Team, Track
- **Validation:** 5-fold Cross-Validation

### Confidence Calculation
- Ensemble predictions from all decision trees
- 95% Confidence Interval = Mean +/- 1.96 x Std Dev

### Saved Artifacts (models/ folder)
- `f1_model.pkl` - Trained Random Forest model
- `Driver_encoder.pkl` - Driver label encoder
- `Team_encoder.pkl` - Team label encoder  
- `Track_encoder.pkl` - Track label encoder

### Limitations
This prediction does not account for:
- Weather conditions
- Car upgrades
- Race incidents/safety cars
- Tyre strategy

---
*Created with F1 2025 Season Data Analysis Pipeline*

## 1. Load Season Data (Race + Sprint)
Load both race and sprint results for accurate championship standings calculation.

In [ ]:
# Load 2025 Season Race and Sprint Results
race_path = project_root / 'data' / 'Formula1_2025Season_RaceResults.csv'
sprint_path = project_root / 'data' / 'Formula1_2025Season_SprintResults.csv'

df_race = load_data(str(race_path))
df_sprint = load_data(str(sprint_path))

df_race = clean_data(df_race)

# Fix sprint data types
df_sprint['Points'] = pd.to_numeric(df_sprint['Points'], errors='coerce').fillna(0)
df_sprint['Position'] = pd.to_numeric(df_sprint['Position'], errors='coerce')

print(f'Race Data: {len(df_race)} entries from {df_race["Track"].nunique()} Grand Prix')
print(f'Sprint Data: {len(df_sprint)} entries from {df_sprint["Track"].nunique()} Sprint events')
print(f'Drivers: {df_race["Driver"].nunique()} | Teams: {df_race["Team"].nunique()}')
print(f'\n2025 Grand Prix Calendar:')
print(list(df_race['Track'].unique()))

## 2. Current Championship Standings (Race + Sprint)
Official FIA standings combining both race and sprint points as per regulations.

In [ ]:
# Calculate Combined Championship Standings (Race + Sprint)
driver_stats = calculate_driver_stats(df_race)
team_stats = calculate_team_stats(df_race)

# Official combined standings
combined_driver_standings = calculate_combined_standings(df_race, df_sprint)
combined_team_standings = calculate_combined_constructor_standings(df_race, df_sprint)

print('=' * 70)
print("         2025 FIA FORMULA 1 DRIVERS' WORLD CHAMPIONSHIP")
print('         (After Qatar GP - Race + Sprint Points)')
print('=' * 70)

standings_display = combined_driver_standings.head(10).copy()
standings_display = standings_display.reset_index()
standings_display.insert(0, 'Pos', range(1, len(standings_display) + 1))
standings_display[['Pos', 'Driver', 'Team', 'Race_Points', 'Sprint_Points', 'Total_Points', 'Race_Wins']]

In [ ]:
# Constructor Championship Standings (Race + Sprint)
print('=' * 70)
print("         2025 FIA FORMULA 1 CONSTRUCTORS' WORLD CHAMPIONSHIP")
print('         (After Qatar GP - Race + Sprint Points)')
print('=' * 70)

constructor_display = combined_team_standings.reset_index()
constructor_display.insert(0, 'Pos', range(1, len(constructor_display) + 1))
constructor_display[['Pos', 'Team', 'Race_Points', 'Sprint_Points', 'Total_Points']]

## 3. Model Training and Accuracy Metrics
Train Random Forest model with detailed accuracy and confidence metrics.

In [ ]:
# Train Random Forest Model with Cross-Validation
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print('Training Random Forest model on 2025 season data...\n')
model, X_test, y_test = train_model(df_race)

# Evaluate model with detailed metrics
metrics, results = evaluate_model(model, X_test, y_test)

# Cross-validation for confidence estimation
X_full, y_full = prepare_features(df_race)
cv_scores = cross_val_score(model, X_full, y_full, cv=5, scoring='neg_mean_absolute_error')
cv_mae = -cv_scores.mean()
cv_std = cv_scores.std()

print('=' * 60)
print('              MODEL PERFORMANCE METRICS')
print('=' * 60)
print(f'''
  Mean Absolute Error (MAE):     {metrics["MAE"]:.2f} positions
  Root Mean Squared Error:       {metrics["RMSE"]:.2f} positions
  R² Score:                      {metrics["R2"]:.3f}
  
  Cross-Validation MAE:          {cv_mae:.2f} ± {cv_std:.2f} positions
  Model Accuracy:                {(1 - cv_mae/10) * 100:.1f}%
  
  Prediction Confidence:         ±{cv_mae + cv_std:.1f} positions (95% CI)
''')
print('=' * 60)

In [ ]:
# Feature Importance Analysis
feature_names = ['Starting Grid', 'Driver', 'Team', 'Track']
importance = pd.DataFrame({
    'Feature': feature_names,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=True)

fig = px.bar(importance, x='Importance', y='Feature', orientation='h',
             title='Feature Importance for Position Prediction',
             color='Importance', color_continuous_scale='Reds')
fig.update_layout(template='plotly_dark')
fig.show()

# Prediction vs Actual scatter
fig = px.scatter(results, x='Actual', y='Predicted',
                 title='Model Validation: Predicted vs Actual Positions',
                 labels={'Actual': 'Actual Position', 'Predicted': 'Predicted Position'})
fig.add_shape(type='line', x0=1, y0=1, x1=20, y1=20, line=dict(dash='dash', color='red'))
fig.update_layout(template='plotly_dark')
fig.show()

## 4. Abu Dhabi GP 2025 - Grid Prediction
Estimated starting grid based on season qualifying performance metrics.

In [ ]:
# Generate Predicted Starting Grid based on Season Qualifying Performance
drivers_teams = df_race.groupby('Driver')['Team'].first().reset_index()

# Calculate average grid position (proxy for qualifying pace)
avg_grid = df_race.groupby('Driver')['Starting Grid'].mean().reset_index()
avg_grid.columns = ['Driver', 'Avg_Grid']

# Calculate grid consistency (standard deviation)
grid_std = df_race.groupby('Driver')['Starting Grid'].std().reset_index()
grid_std.columns = ['Driver', 'Grid_Consistency']

# Merge data
abu_dhabi_grid = drivers_teams.merge(avg_grid, on='Driver').merge(grid_std, on='Driver')
abu_dhabi_grid = abu_dhabi_grid.sort_values('Avg_Grid').reset_index(drop=True)
abu_dhabi_grid['Predicted_Grid'] = range(1, len(abu_dhabi_grid) + 1)

print('=' * 70)
print('         ABU DHABI GP 2025 - PREDICTED STARTING GRID')
print('         Yas Marina Circuit - December 7, 2025')
print('=' * 70)
print('(Based on average qualifying performance this season)\n')
abu_dhabi_grid[['Predicted_Grid', 'Driver', 'Team', 'Avg_Grid', 'Grid_Consistency']].head(20)

In [ ]:
# Prepare Prediction Data with Encoders
from sklearn.preprocessing import LabelEncoder
import pickle

model_dir = project_root / 'models'
encoders = {}
for col in ['Driver', 'Team', 'Track']:
    encoder_path = model_dir / f'{col}_encoder.pkl'
    with open(encoder_path, 'rb') as f:
        encoders[col] = pickle.load(f)

# Create prediction dataframe
abu_dhabi_pred = abu_dhabi_grid.copy()
abu_dhabi_pred['Track'] = 'Abu Dhabi'
abu_dhabi_pred['Starting Grid'] = abu_dhabi_pred['Predicted_Grid']

# Handle track encoding
if 'Abu Dhabi' not in encoders['Track'].classes_:
    print('Note: Abu Dhabi not in training data, using proxy track encoding')
    available_tracks = list(encoders['Track'].classes_)
    abu_dhabi_pred['Track'] = available_tracks[0]

print('Prediction data prepared successfully.')

In [ ]:
# Generate Race Predictions with Confidence Intervals
X_pred = abu_dhabi_pred[['Starting Grid', 'Driver', 'Team', 'Track']].copy()

for col in ['Driver', 'Team', 'Track']:
    le = encoders[col]
    X_pred[col] = X_pred[col].apply(
        lambda x: le.transform([x])[0] if x in le.classes_ else -1
    )

# Ensemble prediction from all trees for confidence interval
all_predictions = np.array([tree.predict(X_pred) for tree in model.estimators_])
predicted_positions = all_predictions.mean(axis=0)
prediction_std = all_predictions.std(axis=0)

# Add to dataframe
abu_dhabi_pred['Predicted_Position'] = predicted_positions
abu_dhabi_pred['Prediction_Std'] = prediction_std
abu_dhabi_pred['Confidence_Lower'] = np.maximum(1, predicted_positions - 1.96 * prediction_std)
abu_dhabi_pred['Confidence_Upper'] = np.minimum(20, predicted_positions + 1.96 * prediction_std)

# Sort by predicted position
final_prediction = abu_dhabi_pred.sort_values('Predicted_Position').reset_index(drop=True)
final_prediction['Final_Position'] = range(1, len(final_prediction) + 1)

print('Predictions generated with 95% confidence intervals.')

## 5. Abu Dhabi GP 2025 - Predicted Race Results
Race classification prediction with confidence intervals and points allocation.

In [ ]:
# Calculate Predicted Points with Confidence Ranges
points_system = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}

final_prediction['Predicted_Points'] = final_prediction['Final_Position'].map(
    lambda x: points_system.get(x, 0)
)

# Calculate confidence percentage (based on prediction std)
final_prediction['Confidence_%'] = (100 - final_prediction['Prediction_Std'] * 10).clip(50, 99).round(1)

print('=' * 80)
print('                 ABU DHABI GP 2025 - PREDICTED RACE RESULTS')
print('                 Yas Marina Circuit - December 7, 2025')
print('=' * 80)

result_display = final_prediction[[
    'Final_Position', 'Driver', 'Team', 'Predicted_Grid', 
    'Predicted_Points', 'Confidence_%'
]].copy()
result_display.columns = ['Pos', 'Driver', 'Team', 'Grid', 'Points', 'Confidence']
result_display['Confidence'] = result_display['Confidence'].apply(lambda x: f'{x}%')
result_display

In [ ]:
# Visualize Predicted Top 10 with Confidence Intervals
top_10 = final_prediction.head(10)
colors = ['gold', 'silver', '#CD7F32'] + ['#E10600'] * 7

fig = go.Figure()

fig.add_trace(go.Bar(
    x=top_10['Driver'],
    y=top_10['Predicted_Points'],
    marker_color=colors,
    text=top_10.apply(lambda x: f"P{x['Final_Position']}<br>{x['Confidence_%']}%", axis=1),
    textposition='outside',
    error_y=dict(
        type='data',
        symmetric=False,
        array=top_10['Prediction_Std'] * 2,
        arrayminus=top_10['Prediction_Std'] * 2,
        visible=True
    )
))

fig.update_layout(
    title='Abu Dhabi GP 2025 - Predicted Top 10 with Confidence Intervals',
    xaxis_title='Driver',
    yaxis_title='Predicted Points',
    template='plotly_dark',
    showlegend=False
)
fig.show()

In [ ]:
# Grid vs Predicted Finish Position with Confidence Bands
fig = go.Figure()

# Add confidence band
fig.add_trace(go.Scatter(
    x=list(final_prediction['Predicted_Grid']) + list(final_prediction['Predicted_Grid'][::-1]),
    y=list(final_prediction['Confidence_Lower']) + list(final_prediction['Confidence_Upper'][::-1]),
    fill='toself',
    fillcolor='rgba(225,6,0,0.2)',
    line=dict(color='rgba(255,255,255,0)'),
    name='95% Confidence'
))

# Add predicted positions
fig.add_trace(go.Scatter(
    x=final_prediction['Predicted_Grid'],
    y=final_prediction['Final_Position'],
    mode='markers+text',
    text=final_prediction['Driver'].apply(lambda x: x.split()[-1][:3].upper()),
    textposition='top center',
    marker=dict(size=12, color='#E10600'),
    name='Predicted'
))

# Add diagonal line
fig.add_trace(go.Scatter(
    x=[1, 20], y=[1, 20],
    mode='lines',
    line=dict(dash='dash', color='gray'),
    name='No change'
))

fig.update_layout(
    title='Grid Position vs Predicted Finish (with 95% Confidence Band)',
    xaxis_title='Grid Position',
    yaxis_title='Predicted Finish Position',
    template='plotly_dark',
    xaxis=dict(range=[0, 21], dtick=2),
    yaxis=dict(range=[0, 21], dtick=2, autorange='reversed')
)
fig.show()

## 6. Championship Impact Analysis
Predicted final championship standings after Abu Dhabi GP (Race + Sprint combined).

In [ ]:
# Calculate Predicted Final Championship Standings (Race + Sprint + Abu Dhabi Prediction)
current_standings = combined_driver_standings[['Total_Points', 'Race_Points', 'Sprint_Points']].copy()
current_standings = current_standings.reset_index()

# Merge with Abu Dhabi predicted points
abu_dhabi_points = final_prediction[['Driver', 'Predicted_Points', 'Confidence_%']]
final_standings = current_standings.merge(abu_dhabi_points, on='Driver', how='left')
final_standings['Predicted_Points'] = final_standings['Predicted_Points'].fillna(0)
final_standings['Confidence_%'] = final_standings['Confidence_%'].fillna(0)

# Calculate final totals
final_standings['Final_Total'] = final_standings['Total_Points'] + final_standings['Predicted_Points']
final_standings = final_standings.sort_values('Final_Total', ascending=False).reset_index(drop=True)
final_standings['Final_Position'] = range(1, len(final_standings) + 1)

# Calculate position change from current standings
current_pos = {d: i+1 for i, d in enumerate(current_standings.sort_values('Total_Points', ascending=False)['Driver'])}
final_standings['Position_Change'] = final_standings['Driver'].map(current_pos) - final_standings['Final_Position']

print('=' * 85)
print('         PREDICTED FINAL 2025 CHAMPIONSHIP STANDINGS')
print('         (After Abu Dhabi GP - Race + Sprint + Predicted Abu Dhabi Points)')
print('=' * 85)

display_standings = final_standings[[
    'Final_Position', 'Driver', 'Total_Points', 'Predicted_Points', 
    'Final_Total', 'Position_Change', 'Confidence_%'
]].head(15).copy()
display_standings.columns = ['Pos', 'Driver', 'Current Pts', 'Abu Dhabi', 'Final Total', 'Change', 'Conf %']
display_standings['Change'] = display_standings['Change'].apply(lambda x: f'+{int(x)}' if x > 0 else (f'{int(x)}' if x < 0 else '-'))
display_standings

In [ ]:
# Visualize Championship Impact - Before vs After Abu Dhabi
top_8_standings = final_standings.head(8)

fig = go.Figure()

fig.add_trace(go.Bar(
    name='Race Points',
    x=top_8_standings['Driver'],
    y=top_8_standings['Race_Points'],
    marker_color='#1E1E1E',
    text=top_8_standings['Race_Points'].astype(int),
    textposition='inside'
))

fig.add_trace(go.Bar(
    name='Sprint Points',
    x=top_8_standings['Driver'],
    y=top_8_standings['Sprint_Points'],
    marker_color='#FF8000',
    text=top_8_standings['Sprint_Points'].astype(int),
    textposition='inside'
))

fig.add_trace(go.Bar(
    name='Abu Dhabi (Predicted)',
    x=top_8_standings['Driver'],
    y=top_8_standings['Predicted_Points'],
    marker_color='#E10600',
    text=top_8_standings['Predicted_Points'].astype(int),
    textposition='inside'
))

fig.update_layout(
    title='Predicted Final Championship - Points Breakdown',
    barmode='stack',
    template='plotly_dark',
    yaxis_title='Total Points',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

## 7. Prediction Summary and Accuracy Report
Comprehensive summary of predictions with confidence metrics and accuracy analysis.

In [ ]:
winner = final_prediction.iloc[0]
p2 = final_prediction.iloc[1]
p3 = final_prediction.iloc[2]
champion = final_standings.iloc[0]

print('=' * 70)
print('             ABU DHABI GP 2025 - PREDICTION SUMMARY')
print('             Yas Marina Circuit - December 7, 2025')
print('=' * 70)

print(f'''
╔══════════════════════════════════════════════════════════════════╗
║                    PREDICTED PODIUM                              ║
╠══════════════════════════════════════════════════════════════════╣
║  🥇 1st: {winner['Driver']:<25} ({winner['Team']})
║         Confidence: {winner['Confidence_%']:.1f}%
║
║  🥈 2nd: {p2['Driver']:<25} ({p2['Team']})
║         Confidence: {p2['Confidence_%']:.1f}%
║
║  🥉 3rd: {p3['Driver']:<25} ({p3['Team']})
║         Confidence: {p3['Confidence_%']:.1f}%
╚══════════════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════════════╗
║              2025 WORLD CHAMPION PREDICTION                       ║
╠══════════════════════════════════════════════════════════════════╣
║  🏆 {champion['Driver']:<30}
║     Final Points: {champion['Final_Total']:.0f}
║     (Current: {champion['Total_Points']:.0f} + Abu Dhabi: {champion['Predicted_Points']:.0f})
╚══════════════════════════════════════════════════════════════════╝

╔══════════════════════════════════════════════════════════════════╗
║                  MODEL ACCURACY METRICS                           ║
╠══════════════════════════════════════════════════════════════════╣
║  Mean Absolute Error:      {metrics['MAE']:.2f} positions
║  R² Score:                 {metrics['R2']:.3f}
║  Cross-Validation MAE:     {cv_mae:.2f} ± {cv_std:.2f} positions
║  Overall Model Accuracy:   {(1 - cv_mae/10) * 100:.1f}%
║  Prediction Confidence:    ±{cv_mae + cv_std:.1f} positions (95% CI)
╚══════════════════════════════════════════════════════════════════╝
''')

print('=' * 70)
print('Note: Predictions based on 2025 season data using Random Forest ML.')
print('Results may vary due to weather, incidents, strategy, and car setup.')
print('=' * 70)

---

## Methodology and Disclaimer

### Model Architecture
- **Algorithm:** Random Forest Classifier (100 estimators)
- **Features:** Starting Grid, Driver, Team, Track
- **Training Data:** 2025 F1 Season (23 races, 459 entries)
- **Validation:** 5-fold Cross-Validation

### Confidence Calculation
- Ensemble predictions from all decision trees
- 95% Confidence Interval = Mean ± 1.96 × Std Dev
- Individual confidence based on prediction variance

### Limitations
This prediction does not account for:
- Weather conditions (rain, temperature)
- Car upgrades and setup changes
- Race incidents and safety cars
- Tyre degradation and strategy
- Driver form and motivation

**Use this prediction for entertainment and analysis purposes only.**

---
*Created with F1 2025 Season Data Analysis Pipeline*